In [55]:

import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt


### This is OpenCV learning project like a CamScanner clone, built from scratch.

In this project i will include all pipeline stages.

The first stage is preprocessing.
It will include resizing, grayscale, bluring, and edge detection.


In [56]:
from pathlib import Path
import os
def contour_detection(img, edged):

    contours, _ = cv.findContours(edged.copy(), cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv.contourArea, reverse=True)
    img_area = img.shape[0] * img.shape[1]
    doc_contour = None
    for c in contours:
        area = cv.contourArea(c)
        if area > 0.90 * img_area:
            continue
        peri = cv.arcLength(c, True)
        approx = cv.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4:
            doc_contour = approx
            break

    return doc_contour

In [57]:
def merge_lines(lines_group):
    if not lines_group:
        return None

    flat = [np.array(l).flatten() for l in lines_group]

    x1 = int(np.mean([l[0] for l in flat]))
    y1 = int(np.mean([l[1] for l in flat]))
    x2 = int(np.mean([l[2] for l in flat]))
    y2 = int(np.mean([l[3] for l in flat]))

    return [x1, y1, x2, y2]


def cluster_lines(lines_group, is_horizontal):
    if not lines_group:
        return []

    positions = []
    for l in lines_group:
        if is_horizontal:
            mid = (l[1] + l[3]) /2
        else:
            mid = (l[0] + l[2]) /2
        positions.append(mid)


    positions = np.array(positions).reshape(-1, 1)

    from sklearn import cluster

    clusters = cluster.DBSCAN(eps=20, min_samples=1).fit(positions)

    merged = []

    for label in set(clusters.labels_):
        group = [lines_group[i] for i, l in enumerate(clusters.labels_) if l == label]
        merged.append(group)


    return merged



def keep_outermost(clustered_groups, is_horizontal):
    if not clustered_groups:
        return None
    merged = [merge_lines(group) for group in clustered_groups]
    merged = [l for l in merged if l is not None]  # filter out any None results
    if len(merged) <= 2:
        return merged
    if is_horizontal:
        sorted_lines = sorted(merged, key=lambda l: (l[1] + l[3] ) / 2)
    else:
        sorted_lines = sorted(merged, key=lambda l: (l[0] + l[2] ) / 2)


    return [sorted_lines[0], sorted_lines[-1]]


def line_intersection(line1, line2):
    x1, y1, x2, y2 = line1
    x3, y3, x4, y4 = line2

    denom = (x1-x2)*(y3-y4) - (y1-y2)*(x3-x4)
    if abs(denom) < 1e-10:
        return None

    t = ((x1 - x3) * (y3 - y4)) / denom
    x = x1 + t * (x2 - x1)
    y = y1 + t * (y3 - y4)

    return [x, y]

def get_corners(h_lines, v_lines):
    corners = []

    for h in h_lines:
        for v in v_lines:
            pt = line_intersection(h, v)
            if pt is not None:
                corners.append(pt)
    return corners




def hough_lines_detection(img, edged):
    debug = img.copy()
    lines = cv.HoughLinesP(
        edged, 1, np.pi / 180, threshold=80, minLineLength=100, maxLineGap=10)

    horizontal = []
    vertical = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = np.degrees(np.arctan2(y2 -y1, x2-x1))
        if abs(angle) < 45:
            horizontal.append(line[0])
        else:
            vertical.append(line[0])

    horizontal_merged = cluster_lines(horizontal, True)
    vertical_merged = cluster_lines(vertical, False)

    horizont = keep_outermost(horizontal_merged, True)
    vertic = keep_outermost(vertical_merged, False)


    debug = img.copy()
    for l in horizont:
        cv.line(debug, (l[0], l[1]), (l[2], l[3]), (0, 255, 0), 2)
        for l in vertic:
            cv.line(debug, (l[0], l[1]), (l[2], l[3]), (0, 0, 255), 2)
    cv.imshow("Final 4 lines", debug)
    cv.waitKey(0)

In [58]:
#GrabCUT

def grabcut_detection(img):
    mask = np.zeros(img.shape[:2], np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    h, w = img.shape[:2]

    margin = int(min(h, w) * 0.02)
    rect = (margin, margin, w - margin*2, h - margin*2)

    cv.grabCut(img, mask, rect, bgd_model, fgd_model, 5, cv.GC_INIT_WITH_RECT)

    mask2 = np.where((mask == cv.GC_FGD) | (mask == cv.GC_PR_FGD), 255, 0).astype('uint8')


    contours, _ = cv.findContours(mask2, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None

    c = max(contours, key=cv.contourArea)

    peri = cv.arcLength(c, True)

    approx = cv.approxPolyDP(c, 0.02 * peri, True)

    if len(approx) != 4:

        for epsilon in [0.03, 0.04, 0.05, 0.08, 0.1]:
            approx = cv.approxPolyDP(c, epsilon*peri, True)
            if len(approx) == 4:
                break

    if len(approx) != 4:
        return None

    return np.array(approx, dtype="float32").reshape(4, 2)



In [59]:
# highest sum is the bottom-right
#lowest sum is the top-left
def order_points(pts):
    pts = np.array(pts, dtype="float32").reshape(4, 2)

    # find centroid
    center = pts.mean(axis=0)

    # compute angle of each point relative to centroid
    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])

    # sort by angle
    sorted_idx = np.argsort(angles)
    sorted_pts = pts[sorted_idx]

    # sorted_pts now goes clockwise or counterclockwise from the "rightmost" point
    # rotate so top-left comes first
    # top-left = point with smallest sum of coordinates
    sums = sorted_pts.sum(axis=1)
    tl_idx = np.argmin(sums)

    # reorder starting from top-left, going clockwise
    ordered = np.roll(sorted_pts, -tl_idx, axis=0)

    return ordered.astype("float32")




In [60]:
def match_points_generation(img):
    points = []
    clone = img.copy()

    def draw_points(event, x, y, flags, param):
        if event == cv.EVENT_LBUTTONDOWN and len(points) < 4:
            points.append([x, y])

            cv.circle(clone, (x, y), 8, (0, 255, 0), -1)

            cv.putText(clone, str(len(points)), (x+12, y), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            if len(points) > 1:
                cv.line(clone, tuple(points[-2]), tuple(points[-1]), (0, 255, 0 ), 1)
            
            cv.imshow("Click 4 corners - press R to reset", clone)

            if len(points) == 4:
                cv.line(clone, tuple(points[-1]), tuple(points[0]), (0, 255, 0), 1)
                cv.imshow("Click 4 corners - press R to reset", clone)

    cv.imshow("Click 4 corners - press R to reset", clone)

    cv.setMouseCallback("Click 4 corners - press R to reset", draw_points)

    while True:

        key = cv.waitKey(1) 

        if key == ord("r"):
            points.clear()
            clone = img.copy()
            cv.putText(clone, "Reset - click 4 corners",
                       (10, 30), cv.FONT_HERSHEY_SIMPLEX,
                       0.7, (0, 0, 255), 2)
            cv.imshow("Click 4 corners - press R to reset", clone)

        elif len(points) == 4:
            cv.waitKey(500)
            break
    try:
        cv.destroyAllWindows("Clock 4 corners - press R to reset")
    except:
        pass
    
    corners = np.array(points, dtype="float32").reshape(4, 2)

    return order_points(corners)

In [ ]:
def find_contour(img):
    height = 500
    ratio = img.shape[0] / height
    img = cv.resize(img, (int(img.shape[1] / ratio), height)) # interpolation is not needed as we are about to blur it anyway.

    gray_img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

    blur_gray_img = cv.GaussianBlur(gray_img, (5, 5), 0)

    low, up = 70, 200
    edged_blur_gray_img = cv.Canny(blur_gray_img, low, up)

    kernel = np.ones((5, 5), np.uint8)

    edged = cv.morphologyEx(edged_blur_gray_img, cv.MORPH_CLOSE, kernel)

    contour = contour_detection(img, edged)

    if contour is not None:
        return contour, ratio

    # corners = hough_lines_detection(img, edged)
    # if corners is not None:
    #     contour = np.array(corners, dtype="float32")
    #     return contour, ratio

    contour = grabcut_detection(img)
    if contour is not None:
        return contour, ratio


    return match_points_generation(img), ratio







In [62]:
def process(image_path: Path):
    img = cv.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not load image at {os.path.join(root, image_path)}")

    orig = img.copy()

    contour, ratio = find_contour(img)
    print("Contour:", contour)
    print("Ratio:", ratio)
    if contour is None:
        print("No contour found")
        return
    rect = order_points(contour) * ratio
    print("Contour after order_points:", rect)

    (tl, tr, br, bl) = rect

    widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))

    width = int(max(widthA, widthB))
    heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))

    height = int(max(heightA, heightB))

    dst = np.array([
        [0, 0],
        [width-1, 0],
        [width-1, height-1],
        [0, height-1]
    ], dtype="float32")

    M = cv.getPerspectiveTransform(rect, dst)
    warped = cv.warpPerspective(orig, M, (width, height))
    cv.imshow("warped", warped)
    cv.waitKey(0)

In [63]:

image_path = Path("../data/IMG_4750.png")
img = cv.imread(image_path)
process(image_path)

Contour: [[129.  72.]
 [484. 118.]
 [637. 280.]
 [ 45. 248.]]
Ratio: 8.568
Contour after order_points: [[1105.272  616.896]
 [4146.912 1011.024]
 [5457.816 2399.04 ]
 [ 385.56  2124.864]]
